# Predicción de Fallas en una Flota de Máquinas Mediante Machine Learning
## Proyecto del curso Machine Learning y Deep Learning Industrial
### Entrega 1: Integración de Datos, Formulación sin Fuga Temporal y Línea Base

**Autores:** Juan Pablo Torres Guillen, Angel David Lugo Lafarga, Jose Pablo Lopez Rodriguez  
**Institución:** Tecnológico de Monterrey  
**Fecha:** Noviembre 2026

---
### Objetivos de la Entrega 1:
1. Integrar las 5 fuentes industriales (telemetría, errores, mantenimientos, fallas y metadatos).
2. Realizar un análisis exploratorio (EDA) que identifique patrones por modelo y firmas pre-falla (48-72h).
3. Analizar la relación causal entre `PdM_maint` y `PdM_failures` para mitigar **fuga de información (data leakage)**.
4. Construir las etiquetas de predicción para un horizonte temporal de $H = 24$ horas (binaria y multiclase).
5. Implementar el pipeline de ingeniería de variables y partición temporal fija.
6. Evaluar modelos de línea base clásica (Regresión Logística y Gradient Boosting) en el conjunto de validación.

In [ ]:
import sys, os
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from data_pipeline import load_raw_data, run_pipeline, generate_imbalance_table
from eda import run_eda
from baseline_models import train_and_evaluate_baselines

print("Entorno inicializado correctamente.")

## 1. Carga e Inspección de Datos Originales
Cargamos los 5 CSVs originales del dataset Microsoft Azure Predictive Maintenance.

In [ ]:
data = load_raw_data("../data/raw")
telemetry = data["telemetry"]
errors = data["errors"]
maint = data["maint"]
failures = data["failures"]
machines = data["machines"]

display(telemetry.head(3))
display(machines.head(3))

## 2. Análisis Causal de Mantenimiento y Fuga de Información
Verificamos si los mantenimientos son consecuencia del fallo o precursores.

In [ ]:
maint_dt = set(zip(maint["datetime"], maint["machineID"], maint["comp"]))
fail_dt = set(zip(failures["datetime"], failures["machineID"], failures["failure"]))
overlap = fail_dt.intersection(maint_dt)
print(f"Total fallas: {len(fail_dt)} | Coincidencia exacta en maint: {len(overlap)} ({len(overlap)/len(fail_dt)*100:.1f}%)")
print("\n-> CONCLUSIÓN: El 97.6% de los mantenimientos son correctivos inmediatos a la falla.")
print("   Usar la presencia de mantenimiento en t generaría fuga de información letal.")
print("   Se debe usar únicamente el tiempo transcurrido desde el último mantenimiento previo.")

## 3. Ejecución del Pipeline de Ingeniería de Variables y Partición Temporal
Ejecutamos el pipeline causal con ventanas móviles cerradas a la derecha y margen de $H=24$h.

In [ ]:
train_df, val_df, test_df, feature_cols = run_pipeline(
    raw_dir="../data/raw",
    processed_dir="../data/processed"
)

print(f"Características generadas ({len(feature_cols)}):", feature_cols)

## 4. Tabla de Desbalance de Clases
Visualización de la proporción de positivos en Train y Validación.

In [ ]:
imbalance_df = pd.read_csv("../data/processed/imbalance_table.csv")
display(imbalance_df)

## 5. Modelado de Línea Base Clásica y Evaluación en Validación
Entrenamiento de Regresión Logística y HistGBDT (LightGBM) sobre el conjunto de validación.

In [ ]:
metrics = train_and_evaluate_baselines(
    processed_dir="../data/processed",
    figures_dir="../figures",
    reports_dir="../reports"
)

## 6. Resultados y Conclusiones
Se generó el reporte oficial `Reporte_Entrega1_PdM.pdf` con todas las figuras y métricas requeridas.